Utilizzando l'API funzionale, crea un modello che accetti tre intressi distinti (ognuno con 8 feature). Ogni ingresso deve passare attraverso un proprio layer Dense da 4 neuroni. Successivamente, unisci tutti e tre i rami e produci un output finale per un problema di regressione (1 neurone, attivazione lineare)

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np
from tensorflow.keras.utils import plot_model

# =============================================================================
# 1. PREPARAZIONE DEI DATI (Dataset Sintetico)
# =============================================================================

# Caricamento dati
n_samples=1000
data1 = np.random.random((n_samples, 8))
data2 = np.random.random((n_samples, 8))
data3 = np.random.random((n_samples, 8))

# Target per regressione: un valore continuo che dipende da una combinazione lineare di data1 e data2, più un po' di rumore.
targets = np.sum(data1, axis=1) + np.mean(data2, axis=1) - np.max(data3, axis=1)

# =============================================================================
# 2. DEFINIZIONE DELL'ARCHITETTURA (API FUNZIONALE)
# =============================================================================

# Definizione dei tre ingressi distinti
input_1 = layers.Input(shape=(8,), name="Ingresso_1")
input_2 = layers.Input(shape=(8,), name="Ingresso_2")
input_3 = layers.Input(shape=(8,), name="Ingresso_3")
# --- RAMO TECNICO ---
# Elaboriamo i dati tecnici con layer densi. 
# Sintassi funzionale: Layer()(Input) crea un legame diretto tra i nodi.

# Ogni ingresso passa attraverso il proprio layer nascosto di elaborazione (dopo aver passato il layer di input)
branch_1 = layers.Dense(4, activation='relu',name="Dense_1")(input_1) # Elaborazione data1
branch_2 = layers.Dense(4, activation='relu',name="Dense_2")(input_2) # Elaborazione data2
branch_3 = layers.Dense(4, activation='relu',name="Dense_3")(input_3) # Elaborazione data3

# potrebbero esserci altri layer di elaborazione per ogni ramo (ma anche non per tutti i rami).

# --- FUSIONE DEI RAMI (CONCATENATE) ---
# Uniamo i tre flussi. 
# Il vettore risultante avrà dimensione 4 (da branch_1) + 4 (da branch_2) + 4 (da branch_3) = 12.
# Questa fase permette alla rete di trovare correlazioni tra i vari input e di combinare le informazioni in modo più efficace.
marged = layers.concatenate([branch_1, branch_2, branch_3])

# Dopo la fusione, possiamo aggiungere altri layer per elaborare ulteriormente le informazioni combinate.
z = layers.Dense(8, activation='relu', name="Dense_Post_Concatenate")(marged)

# Output finale: 1 solo neurone con attivazione Lineare per dare un valore continuo.
output_finale = layers.Dense(1, activation='linear', name="Output")(z)

# =============================================================================
# 3. CREAZIONE E COMPILAZIONE DEL MODELLO
# =============================================================================

# L'oggetto Model definisce i confini del grafo: dove inizia (lista input) e dove finisce.
model = Model(inputs=[input_1, input_2, input_3], outputs=output_finale)

# Compiliamo specificando l'ottimizzatore e la funzione di perdita per classificazione binaria.
model.compile(optimizer='adam', 
              loss='mse', 
              metrics=['mae'])

# Visualizza la struttura: noterai come i due input iniziali convergono verso il layer 'concatenate'.
model.summary()

# =============================================================================
# 4. ADDESTRAMENTO
# =============================================================================

# Quando il modello ha più input, il parametro 'x' deve ricevere una LISTA di array.
# L'ordine nella lista deve corrispondere esattamente all'ordine definito in Model(inputs=[...]).
model.fit(
    x=[data1, data2, data3], 
    y=targets, 
    epochs=10, 
    batch_size=32,
    validation_split=0.2
)

print("\nModello addestrato con successo gestendo input eterogenei, regressione a 3 ingressi.")

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Ingresso_1          │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Ingresso_2          │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Ingresso_3          │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_1 (Dense)     │ (None, 4)         │         36 │ Ingresso_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_2 (Dense)     │ (None, 4)         │         36 │ Ingresso_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_3 (Dense)     │ (None, 4)         │         36 │ Ingresso_3[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 12)        │          0 │ Dense_1[0][0],    │
│ (Concatenate)       │                   │            │ Dense_2[0][0],    │
│                     │                   │            │ Dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_Post_Concate… │ (None, 8)         │        104 │ concatenate_4[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output (Dense)      │ (None, 1)         │          9 │ Dense_Post_Conca… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 221 (884.00 B)

 Trainable params: 221 (884.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 12.7425 - mae: 3.4578 - val_loss: 10.2235 - val_mae: 3.0897
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 8.3692 - mae: 2.7635 - val_loss: 5.6102 - val_mae: 2.2385
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 4.0705 - mae: 1.8378 - val_loss: 1.9612 - val_mae: 1.2088
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.3404 - mae: 0.9472 - val_loss: 0.6097 - val_mae: 0.6350
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6984 - mae: 0.6748 - val_loss: 0.5812 - val_mae: 0.6217
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6542 - mae: 0.6548 - val_loss: 0.5489 - val_mae: 0.6061
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6311 - mae: 0.6436 - val_loss: 0.5292 - val_mae: 0.5957
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6116 - mae: 0.6334 - val_loss: 0.5135 - val_mae: 0.5864
Epoch 9/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.5916 - ma